# 02. Base model (π_ref)

Stage 2 of `docs/codeagents/research_plan.md` (§4, §11 stage 2): GPT-2 next-item recommender trained with sampled softmax + logQ, early stopping on val NDCG@10, compared with MostPop / Repeat and with the published SASRec numbers (Yambda-50M, Listen+: NDCG@10 = 0.0748, Recall@10 = 0.0325, Recall@100 = 0.1026, compare with the `paper_*` columns).

Logic lives in `recsys_rl.model`, `recsys_rl.train`, `recsys_rl.evaluate`. Needs a GPU runtime. Training resumes from `runs/<run_name>/last.pt` on Drive, so after a disconnect just rerun the notebook. Tables and plots for the report go to `results/stage2/` on Drive (download that folder into the repo's `results/`). The same run from a terminal: `uv run yambda-train [key=value ...]`.

In [ ]:
# Bootstrap: each Colab notebook runs in its own runtime, so repeat the setup from 00.
# Data must already be on Drive (00_colab_setup.ipynb downloads it). Locally: cd to repo root.
import os
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["RECSYS_DATA_ROOT"] = "/content/drive/MyDrive/tlab-recsys"
    token = (userdata.get("GH_TOKEN") or "").strip()
    if not token:
        raise RuntimeError("Colab secret GH_TOKEN is empty, see notebooks/README.md")
    repo_url = f"https://{token}@github.com/miont/tlab-2026-recsys-task2.git"
    !git clone -q {repo_url} /content/repo || git -C /content/repo pull -q
    if not Path("/content/repo/pyproject.toml").exists():
        raise RuntimeError("git clone failed: check that GH_TOKEN can read the repository")
    %cd /content/repo
    !pip install -q uv
    !uv pip install --system -q -e .
    sys.path.insert(0, "/content/repo/src")  # the editable .pth is only read at startup
elif not Path("configs").exists():
    os.chdir(Path.cwd().parent)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch

from recsys_rl import evaluate
from recsys_rl.config import load_config, parse_overrides
from recsys_rl.data import prepare
from recsys_rl.model import load_model
from recsys_rl.train import amp_dtype, train
from recsys_rl.utils import get_device, save_figure

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_float_precision(4)

device = get_device()
print(device, torch.cuda.get_device_name() if device.type == "cuda" else "")
cfg = load_config("configs/base.yaml", run_name="base")
prepared = prepare(cfg)  # built once on Drive by 01_eda / the first run, then cached
print(prepared.bounds, len(prepared.items), "items,", len(prepared.window_starts), "windows")
RESULTS = cfg.results_dir / "stage2"  # CSV / PNG for the report, on Drive in Colab
RESULTS.mkdir(parents=True, exist_ok=True)

## Baselines

MostPop: Listen+ counts over the last 7 days before the context cutoff. Repeat: the user's most recently listened distinct tracks, padded with MostPop. `exclude_seen=True` is the repeat-filter evaluation mode (already heard tracks are not recommended); it is always reported separately.

In [ ]:
COLUMNS = [
    "split",
    "method",
    "exclude_seen",
    "users",
    "ndcg@10",
    "recall@10",
    "recall@100",
    "map@10",
    "paper_ndcg@10",
    "paper_recall@10",
    "paper_recall@100",
    "coverage@10",
]
baselines = evaluate.results_table(
    prepared,
    cfg.data,
)
baselines.write_csv(RESULTS / "baselines.csv")
baselines.select(COLUMNS)

## Training

`base`: next-item over all listens, with feedback-feature embeddings (played-ratio bin, length bin, organic, like/dislike). Overrides use the CLI syntax, e.g. `"train.lr=3e-4"`.

`BASE_EPOCHS`: the first run used 5 epochs (`configs/base.yaml`) and val NDCG@10 was still growing at the last step without early stopping, so `base` is continued to 8 epochs. The run resumes from `runs/base/last.pt`; the cosine LR schedule is recomputed over the new total, so the LR jumps from ≈0 back to ≈30% of the peak at the resume point (a warm restart). `best.pt` is replaced only if val NDCG@10 improves, and patience = 5 evaluations can stop the run early. From a terminal: `uv run yambda-train train.epochs=8`. The reward models were trained from the 5-epoch encoder; that only affects their initialisation, π_ref for RL is the new `best.pt`.

In [ ]:
def run(run_name: str, *overrides: str):
    overrides = parse_overrides([f"run_name={run_name}", *overrides])
    run_cfg = load_config("configs/base.yaml", **overrides)
    best = train(run_cfg, prepared, device)
    return run_cfg, load_model(best, device)


def plot_log(*run_names: str, name: str):
    fig, (ax_loss, ax_ndcg) = plt.subplots(1, 2, figsize=(12, 4))
    for run in run_names:
        log = pl.read_ndjson(cfg.run_dir.parent / run / "log.jsonl")
        ax_loss.plot(log["step"], log["loss"], label=run)
        ax_ndcg.plot(log["step"], log["val_ndcg@10"], label=run)
    ax_loss.set(xlabel="step", ylabel="sampled-softmax loss")
    ax_ndcg.set(xlabel="step", ylabel="val NDCG@10")
    ax_ndcg.legend()
    save_figure(fig, RESULTS / f"{name}.png")
    plt.show()


BASE_EPOCHS = 8
cfg, model = run("base", f"train.epochs={BASE_EPOCHS}")
plot_log("base", name="training_base")

In [ ]:
results = evaluate.results_table(
    prepared,
    cfg.data,
    models={"gpt2_base": model},
    amp_dtype=amp_dtype(device, cfg.train.amp),
)
results.write_csv(cfg.run_dir / "results.csv")
results.write_csv(RESULTS / "base.csv")
results.select(COLUMNS)

## Examples of recommendations

Yambda is anonymised: tracks, artists and albums are numeric ids, there are no titles. So each example shows the user's last listens (played ratio, like/dislike, organic) and the top-10 of the model, Repeat and MostPop, tagged with artist / album, train popularity rank (`pop_rank`, 1 = most listened), how many times the user heard the track before the cutoff (`heard`), whether its artist / album was heard, and `target` (how many Listen+ events of the val day it got). `targets` gives the rank of every true val Listen+ track in the model's full ranking. The profile table aggregates the same tags over all val users: the share of already heard tracks, unheard tracks of known artists and new artists in the top-10.

In [ ]:
meta = evaluate.load_item_meta(cfg.raw_dir, prepared.items)
idx_to_item = evaluate.idx_to_item_array(prepared.items)
val_set = evaluate.build_eval_set(
    prepared.store, prepared.bounds, "val", cfg.data.max_seq_len, cfg.data.listen_plus_threshold
)
rng = np.random.default_rng(0)
for position in rng.choice(val_set.n_users, size=3, replace=False):
    for exclude_seen in (False, True):
        ex = evaluate.explain_user(
            model, prepared.store, val_set, position, idx_to_item, meta, exclude_seen=exclude_seen
        )
        print(f"=== uid {ex.uid}, exclude_seen={exclude_seen}")
        if not exclude_seen:
            display(ex.history)
        display(ex.recs.filter(pl.col("method") == "model"))
        if not exclude_seen:
            display(ex.recs.filter(pl.col("method") != "model"))
    display(ex.targets)

In [ ]:
profile = pl.DataFrame(
    [
        {"method": name, **evaluate.recs_profile(recs, val_set, prepared.store, meta)}
        for name, recs in {
            "MostPop": evaluate.recommend_most_pop(prepared.store, val_set),
            "Repeat": evaluate.recommend_repeat(prepared.store, val_set),
            "gpt2_base": evaluate.recommend_model(
                model,
                prepared.store,
                val_set,
                idx_to_item,
                amp_dtype=amp_dtype(device, cfg.train.amp),
            ),
            "gpt2_base (exclude_seen)": evaluate.recommend_model(
                model,
                prepared.store,
                val_set,
                idx_to_item,
                exclude_seen=True,
                amp_dtype=amp_dtype(device, cfg.train.amp),
            ),
        }.items()
    ]
)
profile.write_csv(RESULTS / "recs_profile.csv")
profile

## Ablations (stage 2.3, optional)

- `no_feedback`: item IDs only, no feedback-feature embeddings;
- `listen_plus`: next-positive target, only Listen+ listens contribute to the loss (inputs unchanged).

The ablations keep 5 epochs, so for a like-for-like comparison with the extended `base` add `f"train.epochs={BASE_EPOCHS}"` to their overrides. Selection is by val; the test rows are for the report.

In [ ]:
models = {"gpt2_base": model}
for name, overrides in {
    "no_feedback": ["model.use_feedback_tokens=false"],
    "listen_plus": ["train.target=listen_plus"],
}.items():
    models[name] = run(name, *overrides)[1]
plot_log("base", *list(models)[1:], name="training_ablations")
ablations = evaluate.results_table(
    prepared,
    cfg.data,
    models=models,
    amp_dtype=amp_dtype(device, cfg.train.amp),
)
ablations.write_csv(RESULTS / "ablations.csv")
ablations.select(COLUMNS)